# Enterprise RAG Pipeline - UK Visa Documents


This notebook builds a beginner-friendly **enterprise-style RAG pipeline** over multiple UK visa PDF documents.

The word "enterprise" here does not mean the project is finished or production-ready. It means the notebook is moving from one PDF to a more realistic setup where several documents are loaded, indexed, searched, and used together.

We are using free/local Hugging Face models because paid hosted model services cost money. At the beginning, the goal is to understand the RAG architecture clearly before spending money on stronger models.

The full flow is:

1. Load multiple PDFs.
2. Clean their text.
3. Split all pages into chunks.
4. Convert chunks into embeddings.
5. Store the embeddings in FAISS.
6. Retrieve the most relevant chunks for a question.
7. Pass those chunks to a local generation model.
8. Return an answer with source evidence.

This notebook also fixes the earlier bug where the answer function tried to call a model variable that did not exist. Now the notebook creates one local generator and uses it consistently.


## Phase 1 - Load And Index Documents


### Step 1 - Import Libraries


Notes:

This cell imports the tools needed for the full RAG pipeline.

The imports are grouped by purpose:

- file and text utilities help us locate PDFs and clean text,
- PDF loaders extract text from the visa documents,
- text splitters break long pages into smaller chunks,
- embedding models convert chunks into vectors,
- FAISS stores and searches the vectors,
- memory stores the conversation history,
- Hugging Face loads the free local answer model.

A RAG pipeline is easier to understand when you see it as separate responsibilities: load, clean, chunk, embed, retrieve, generate.


In [1]:
import re
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.memory import ConversationBufferMemory
from transformers import pipeline

### Step 2 - Load All PDFs


Notes:

This step loads every PDF in the `Data` folder.

Instead of hard-coding one file, the notebook searches for all files ending in `.pdf`. This makes the pipeline more general. If you add another visa PDF later, the same code can include it automatically.

For each PDF:

1. `PyPDFLoader` reads the pages.
2. Each page becomes a LangChain `Document`.
3. The source file name is stored in metadata.
4. All pages are added into one list called `all_documents`.

Metadata is important because once all PDFs are mixed together, we still need to know which file and page each answer came from.


In [2]:
candidate_folders = [
    Path("Data"),
    Path("RAG-Experiments/Data"),
]

pdf_folder = next((folder for folder in candidate_folders if folder.exists()), None)
if pdf_folder is None:
    raise FileNotFoundError("Could not find the PDF folder. Expected Data/ or RAG-Experiments/Data/.")

all_documents = []

for pdf_path in sorted(pdf_folder.glob("*.pdf")):
    print(f"Loading: {pdf_path.name}")
    loader = PyPDFLoader(str(pdf_path))
    all_documents.extend(loader.load())

print(f"Loaded {len(all_documents)} pages from {len(list(pdf_folder.glob('*.pdf')))} PDFs")

Loading: Global-Talent-Visa.pdf
Loading: Graduate-Visa-Route.pdf
Loading: Skilled-Worker-Visa.pdf
Loading: Standard-Visitor-Route.pdf
Loaded 102 pages from 4 PDFs


### Step 3 - Inspect Loaded Documents


Notes:

This cell checks that the PDFs loaded correctly.

Before building embeddings, always inspect the data. You want to confirm:

- how many pages were loaded,
- which PDF files were included,
- whether the extracted text looks readable,
- whether metadata contains useful source information.

This is a basic but important debugging step. If the source documents are wrong here, every later step will be built on the wrong foundation.


In [3]:
print(f"Total loaded pages: {len(all_documents)}")
print("Sample metadata:", all_documents[0].metadata)
print("Sample text:")
print(all_documents[0].page_content[:700])

Total loaded pages: 102
Sample metadata: {'source': 'Data/Global-Talent-Visa.pdf', 'page': 0}
Sample text:
Apply for the Global
Talent visa
1. Overview
You can apply for a Global Talent visa to work in the UK if you’re a leader or
potential leader in one of the following ﬁelds:
academia or research
arts and culture
digital technology
You must also be at least 18 years old.
Before you apply for a visa
The way you apply for a Global Talent visa depends on your
circumstances. 
If you’ve won an eligible prestigious prize, you can apply for the visa
directly.
If you’ve not won an eligible prestigious prize, you’ll need to get an
endorsement ﬁrst to prove that you’re a leader or potential leader in your
ﬁeld.
If you’re not eligible for a Global Talent visa, there are other ways to
work in the UK (/chec


### Step 4 - Clean PDF Text


Notes:

PDF text often contains repeated print headers, page numbers, URLs, and awkward spacing.

This step cleans each page before chunking. Cleaner text usually creates better chunks, better embeddings, and better retrieval.

The cleaning function is intentionally simple:

- convert line breaks into spaces,
- remove repeated GOV.UK print patterns,
- remove some URL/page-count noise,
- collapse extra whitespace.

For beginner projects, simple cleaning is enough. Later, you can make this more advanced by removing boilerplate sections, tables of contents, or duplicated navigation text.


In [4]:
def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\d{2}/\d{2}/\d{4},\s*\d{2}:\d{2}\s*Print.*?GOV\.UK", " ", text)
    text = re.sub(r"https://www\.gov\.uk/[^\s]+\s+\d+/\d+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

for doc in all_documents:
    doc.page_content = clean_text(doc.page_content)

print(all_documents[0].page_content[:700])

Apply for the Global Talent visa 1. Overview You can apply for a Global Talent visa to work in the UK if you’re a leader or potential leader in one of the following ﬁelds: academia or research arts and culture digital technology You must also be at least 18 years old. Before you apply for a visa The way you apply for a Global Talent visa depends on your circumstances. If you’ve won an eligible prestigious prize, you can apply for the visa directly. If you’ve not won an eligible prestigious prize, you’ll need to get an endorsement ﬁrst to prove that you’re a leader or potential leader in your ﬁeld. If you’re not eligible for a Global Talent visa, there are other ways to work in the UK (/check


### Step 5 - Chunk All Documents


Notes:

Chunking turns long pages into smaller searchable pieces.

This matters because a user question usually relates to one part of a document, not an entire PDF. Smaller chunks let the retriever find the most relevant evidence more precisely.

The settings used here are:

- `chunk_size=500`: each chunk is around 500 characters,
- `chunk_overlap=50`: neighboring chunks share a little text.

The overlap helps preserve meaning when a sentence or section sits near a chunk boundary.


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

docs = text_splitter.split_documents(all_documents)
print(f"Created {len(docs)} chunks")

Created 367 chunks


### Step 6 - Generate Embeddings


Notes:

Embeddings convert text into vectors.

A vector is a list of numbers that represents the meaning of text. Once text is represented as vectors, we can search by meaning instead of only exact words.

This is what allows a question like "Can this visa lead to settlement?" to retrieve text that may say "route to indefinite leave to remain" even if the words are not exactly the same.

We use `sentence-transformers/all-MiniLM-L6-v2` because it is free, lightweight, and suitable for learning.


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/var/folders/7f/3s2chqx942sfr2r40nb5vhww0000gn/T/ipykernel_36487/1474760240.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(


### Step 7 - Create FAISS Vector Store


Notes:

This step creates the vector database.

FAISS stores every chunk together with its embedding. When a question comes in, FAISS can quickly compare the question vector with all chunk vectors and return the closest matches.

Think of FAISS as the search engine for your RAG system. The documents are the knowledge, embeddings are the searchable representation, and FAISS is the place where search happens.


In [7]:
vectorstore = FAISS.from_documents(
    docs,
    embeddings,
)

### Step 8 - Create Retriever


Notes:

The retriever is the part of the pipeline that asks FAISS for relevant chunks.

The setting `k=5` means the retriever returns the five closest chunks for each question.

Choosing `k` is a balance:

- too low may miss useful evidence,
- too high may give the model too much text,
- small local models usually work better with focused context.

For learning, `k=5` is a sensible starting point.


In [8]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

### Step 9 - Test Retrieval Across All Documents


Notes:

This test checks whether retrieval works across multiple PDFs.

At this point, the model is not generating an answer yet. We are only asking: "Can the system find relevant evidence?"

This is a key RAG debugging habit. If retrieval returns the wrong chunks, the final answer will not be reliable. Always inspect retrieved chunks before blaming the language model.


In [9]:
query = "What is the best visa route to settle in the UK for someone holding a Master's degree?"
retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n{'=' * 60}")
    print(f"Chunk {i}")
    print(f"{'=' * 60}")
    print("Source:", doc.metadata)
    print("Content:")
    print(doc.page_content)


Chunk 1
Source: {'source': 'Data/Graduate-Visa-Route.pdf', 'page': 0}
Content:
Graduate visa 1. Overview A Graduate visa gives you permission to stay in the UK for at least 18 months after successfully completing an eligible course in the UK. You must be in the UK when you apply. Eligibility You can apply for a Graduate visa if all of the following are true: you’re in the UK your current visa is a Student visa or Tier 4 (General) student visa (/student-visa) you studied a UK bachelor’s degree, postgraduate degree or other eligible course for a minimum period of time with

Chunk 2
Source: {'source': 'Data/Skilled-Worker-Visa.pdf', 'page': 10}
Content:
professionals not elsewhere classiﬁed 2162: other researchers, unspeciﬁed discipline 2311: higher education teaching professionals If this applies to you, check how much you’ll need to be paid (/government/publications/skilled-worker-visa-eligible-salary-if-youre-under-26- studying-training-or-in-a-postdoctoral-role) to qualify for this v

## Phase 2 - Local Conversational RAG


### Step 1 - Initialize Memory


Notes:

Memory stores previous turns in the conversation.

In a simple one-question RAG pipeline, every question is independent. In a chat-style pipeline, the user might ask follow-up questions like "What about the fees?" or "Can this route lead to settlement?"

Memory helps keep track of earlier messages so the system can build a more natural conversation.

Beginner note: memory does not replace retrieval. The answer should still be grounded in retrieved document chunks. Memory only helps preserve conversational context.


In [10]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
)

### Step 2 - Load Free Local Generation Model


Notes:

This cell loads the model that writes the final answer.

We are using `google/flan-t5-base` through Hugging Face. It is free and useful for learning, but it is smaller and less powerful than paid hosted models.

Because it is small, we help it by:

- retrieving only the most relevant chunks,
- keeping the context short,
- writing clear prompts,
- asking it to answer only from the provided evidence.

This keeps the project affordable while you learn the RAG workflow.


In [11]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
)

Device set to use mps:0


### Step 3 - Build Conversational RAG Function


Notes:

This function connects all the pieces into one reusable RAG workflow.

When you call `ask_question(query)`, the function does the following:

1. Sends the question to the retriever.
2. Retrieves the most relevant document chunks.
3. Combines the top chunks into a context block.
4. Builds a prompt that tells the model to answer from that context.
5. Calls the local Hugging Face generator.
6. Saves the user question and model answer into memory.
7. Returns the answer, sources, and retrieved chunks.

This is the heart of the notebook. Everything before this prepares the data. This function actually runs the RAG loop.


In [12]:
def ask_question(query):
    retrieved_docs = retriever.invoke(query)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs[:3]
    )[:1800]

    chat_history = memory.load_memory_variables({}).get("chat_history", [])

    prompt = f"""
You are an enterprise AI assistant answering questions from UK visa documents.
Use only the provided context to answer.
If the answer is not in the context, say: I could not find the answer in the documents.

Chat history:
{chat_history}

Context:
{context}

Question: {query}

Answer:
"""

    response = generator(
        prompt,
        max_new_tokens=180,
        do_sample=False,
    )

    answer = response[0]["generated_text"]

    memory.save_context(
        {"input": query},
        {"output": answer},
    )

    return {
        "answer": answer,
        "sources": [doc.metadata for doc in retrieved_docs],
    }

### Step 4 - Ask A Question


Notes:

This cell tests the complete pipeline.

Now the system performs both parts of RAG:

- retrieval finds relevant visa document chunks,
- generation turns those chunks into a readable answer.

When reviewing the output, check the answer and the source chunks together. A good RAG answer should be supported by the retrieved evidence.


In [13]:
result = ask_question("What is the best visa route for someone with a Master's degree who wants to settle in the UK?")

print("Answer:")
print(result["answer"])

print("\nSources:")
for source in result["sources"]:
    print(source)

Answer:
Graduate visa 1. Overview A Graduate visa gives you permission to stay in the UK for at least 18 months after successfully completing an eligible course in the UK. You must be in the UK when you apply. Eligibility You can apply for a Graduate visa if all of the following are true: you’re in the UK your current visa is a Student visa or Tier 4 (General) student visa (/student-visa) you studied a UK bachelor’s degree, postgraduate degree or other eligible course for a minimum period of time with professionals not elsewhere classified 2162: other researchers, unspecified discipline 2311: higher education teaching professionals If this applies to you, check how much you’ll need to be paid (/government/publications/skilled-worker-visa-eligible-salary-if

Sources:
{'source': 'Data/Graduate-Visa-Route.pdf', 'page': 0}
{'source': 'Data/Skilled-Worker-Visa.pdf', 'page': 10}
{'source': 'Data/Graduate-Visa-Route.pdf', 'page': 1}
{'source': 'Data/Global-Talent-Visa.pdf', 'page': 1}
{'sourc

## Phase 3 - Streamlit Chat UI


### Step 1 - Install Streamlit

Notes:

Streamlit lets us turn the notebook idea into a small web app.

The notebook is good for learning each step. The Streamlit app is better for interacting with the pipeline like a simple chatbot.

The app file is `app.py`. It uses the same core RAG ideas:

- load PDFs from the `Data` folder,
- build embeddings and a FAISS index,
- retrieve evidence for a question,
- use the free local Hugging Face model to generate an answer,
- show sources and retrieved evidence in the page.

To run the app from the terminal, use:

```bash
cd RAG-Experiments
streamlit run app.py
```

If port 8501 is busy, run it on another port, for example:

```bash
streamlit run app.py --server.port 8502
```


In [14]:
!pip install streamlit